# Phase 2: Data Understanding & Structural Audit
**Project Aftershock: Regional Seismic Risk Triage**

This notebook performs native EDA and quality checks on the USGS earthquake feed without using pandas or numpy.

In [1]:
import json
import requests

# Fetch raw sample from USGS API
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "starttime": "2024-01-01",
    "endtime": "2024-01-05",
    "minmagnitude": 2.5
}
res = requests.get(url, params=params)
data = res.json()
features = data.get("features", [])
print(f"Total features extracted for audit: {len(features)}")

Total features extracted for audit: 334


## 1. Recursive Structural Leaf Walker

In [2]:
def walk_structure(obj, path="root"):
    """Recursively traverses a JSON structure and prints leaf data types and paths."""
    if isinstance(obj, dict):
        for k, v in obj.items():
            walk_structure(v, f"{path}.{k}")
    elif isinstance(obj, list):
        if len(obj) > 0:
            walk_structure(obj[0], f"{path}[0]")
        else:
            print(f"{path}: list (empty)")
    else:
        print(f"{path} -> {type(obj).__name__} (Sample: {obj})")

if features:
    print("--- Structural Audit of First Record ---")
    walk_structure(features[0])

--- Structural Audit of First Record ---
root.type -> str (Sample: Feature)
root.properties.mag -> float (Sample: 4.5)
root.properties.place -> str (Sample: Fiji region)
root.properties.time -> int (Sample: 1704411730828)
root.properties.updated -> int (Sample: 1710020367040)
root.properties.tz -> NoneType (Sample: None)
root.properties.url -> str (Sample: https://earthquake.usgs.gov/earthquakes/eventpage/us6000m1vr)
root.properties.detail -> str (Sample: https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000m1vr&format=geojson)
root.properties.felt -> NoneType (Sample: None)
root.properties.cdi -> NoneType (Sample: None)
root.properties.mmi -> NoneType (Sample: None)
root.properties.alert -> NoneType (Sample: None)
root.properties.status -> str (Sample: reviewed)
root.properties.tsunami -> int (Sample: 0)
root.properties.sig -> int (Sample: 312)
root.properties.net -> str (Sample: us)
root.properties.code -> str (Sample: 6000m1vr)
root.properties.ids -> str (Sample: ,us6000m1

## 2. Native Loop Accumulators (Min / Max / Mean)

In [3]:
mag_sum = 0.0
mag_count = 0
mag_min = float('inf')
mag_max = float('-inf')

depth_sum = 0.0
depth_count = 0
depth_min = float('inf')
depth_max = float('-inf')

for f in features:
    m = f.get("properties", {}).get("mag")
    if m is not None:
        m_val = float(m)
        mag_sum += m_val
        mag_count += 1
        if m_val < mag_min:
            mag_min = m_val
        if m_val > mag_max:
            mag_max = m_val
            
    coords = f.get("geometry", {}).get("coordinates", [])
    if len(coords) >= 3 and coords[2] is not None:
        d_val = float(coords[2])
        depth_sum += d_val
        depth_count += 1
        if d_val < depth_min:
            depth_min = d_val
        if d_val > depth_max:
            depth_max = d_val

print(f"Magnitude -> Min: {mag_min:.2f}, Max: {mag_max:.2f}, Mean: {mag_sum/mag_count:.2f}")
print(f"Depth (km) -> Min: {depth_min:.2f}, Max: {depth_max:.2f}, Mean: {depth_sum/depth_count:.2f}")

Magnitude -> Min: 2.50, Max: 7.50, Mean: 4.08
Depth (km) -> Min: 0.00, Max: 659.94, Mean: 58.47


## 3. Missing Value Rates & Event-Type Contamination Audit

In [4]:
target_fields = ['felt', 'cdi', 'mmi', 'alert', 'nst', 'dmin', 'gap']
null_counts = {field: 0 for field in target_fields}
type_counts = {}
total = len(features)

for f in features:
    props = f.get("properties", {})
    
    for field in target_fields:
        if props.get(field) is None:
            null_counts[field] += 1
            
    ev_type = props.get("type", "unknown")
    type_counts[ev_type] = type_counts.get(ev_type, 0) + 1

print("=== Missing Value Rates ===")
for field, count in null_counts.items():
    pct = (count / total) * 100 if total > 0 else 0
    print(f"{field:<10}: {count:>4}/{total} null ({pct:.2f}%)")

print("\n=== Event Type Breakdown ===")
for ev_type, count in type_counts.items():
    pct = (count / total) * 100 if total > 0 else 0
    print(f"{ev_type:<15}: {count:>4} events ({pct:.2f}%)")

=== Missing Value Rates ===
felt      :  284/334 null (85.03%)
cdi       :  284/334 null (85.03%)
mmi       :  315/334 null (94.31%)
alert     :  323/334 null (96.71%)
nst       :   20/334 null (5.99%)
dmin      :   24/334 null (7.19%)
gap       :   20/334 null (5.99%)

=== Event Type Breakdown ===
earthquake     :  333 events (99.70%)
mining explosion:    1 events (0.30%)
